# chap06 · sec02 — 이미지로 영어 듣기 문제 만들기 (수정본)

원본 `image_quiz.ipynb` 를 고친 노트북입니다. 원본은 그대로 두었습니다.

**하는 일**: `../data/images` 의 jpg 사진마다 GPT-4o에게 한국어 + 영어(토익 리스닝 스타일) 4지선다 문제를 만들게 하고,
- 문제 전문 → `image_quiz_eng.md` (VS Code 미리보기로 사진과 함께 보기)
- 영어 문제만 → `image_quiz_eng.json` (다음 노트북 `tts.ipynb` 가 읽어서 mp3로 만듦)

**원본과 달라진 점**

| 원본의 문제 | 이 노트북 |
|---|---|
| `print("failed\n" + e)` → API 오류 한 번에 `TypeError` 로 셀 전체가 멈춤 | `f"{type(e).__name__}: {e}"` 로 출력하고 다음 시도로 넘어감 |
| 3번 실패하면 `raise` → 반복문 전체가 멈춤 | `(None, None, False)` 를 돌려주고 그 사진만 건너뜀 |
| `"Listening:"` 으로 검사하고 `'Listening: '` 로 잘라 `IndexError` 가능 | 같은 정규식 하나로 검사와 추출을 함께 |
| PNG도 `data:image/jpeg` 로 표시 | `mimetypes` 로 형식을 자동으로 판단 |
| `glob` 결과 순서가 PC마다 다를 수 있음 | `sorted()` 로 정렬해 번호 고정 |
| 재귀 호출로 재시도 | `for` 반복문으로 재시도 (몇 번째 시도인지 보임) |

**실행 전 확인**
1. 커널이 Python 3.10 가상환경인지 (`openai`, `python-dotenv` 설치)
2. 상위 폴더 어딘가의 `.env` 에 `OPENAI_API_KEY=...` 가 있는지
3. 이 노트북이 `chap06/sec02` 폴더에 있는지 (`../data/images` 기준)

셀은 **위에서부터 차례로** 실행합니다. 셀 1~5는 준비(정의)만 하고 API를 부르지 않습니다. **셀 6에서 실제로 API를 부릅니다**(사진 8장 → 최소 8번).

## 셀 1 — 라이브러리 가져오기와 기본 설정

필요한 도구를 가져오고, API 키를 읽어 클라이언트를 만듭니다. 모델 이름과 사진 폴더처럼 **바뀔 수 있는 값은 맨 위에 상수로** 모아 둡니다.

<details><summary>줄번호 붙은 코드 보기 (설명 표와 맞춰 볼 때)</summary>

```text
  1  import base64
  2  import json
  3  import mimetypes
  4  import os
  5  import re
  6  from glob import glob
  7  from dotenv import load_dotenv
  8  from openai import OpenAI
  9  
 10  load_dotenv()
 11  client = OpenAI()
 12  MODEL = "gpt-4o"
 13  IMAGE_DIR = "../data/images"          # 노트북이 sec02 폴더에 있다는 전제 (VS Code 기본 동작)
```
</details>

| 줄 | 설명 |
|---:|---|
| 1 | `import base64` — 사진 파일(바이트)을 글자(base64 문자열)로 바꾸는 표준 라이브러리. JSON 요청에는 글자만 넣을 수 있어서 필요합니다. |
| 2 | `import json` — 영어 문제 목록(리스트·딕셔너리)을 `.json` 파일로 저장하는 표준 라이브러리. |
| 3 | `import mimetypes` — 파일 확장자로 형식 이름표(MIME)를 알아내는 표준 라이브러리. `.jpg` → `image/jpeg`, `.png` → `image/png`. |
| 4 | `import os` — 경로에서 파일 이름만 꺼낼 때(`os.path.basename`) 씁니다. |
| 5 | `import re` — 정규식(글자 패턴 찾기) 표준 라이브러리. 답변에서 영어 문제 부분만 정확히 잘라 낼 때 씁니다. |
| 6 | `from glob import glob` — `*.jpg` 같은 패턴에 맞는 파일 경로들을 리스트로 돌려주는 함수. |
| 7 | `from dotenv import load_dotenv` — `.env` 파일을 읽어 환경 변수로 올리는 함수. 설치 이름은 `python-dotenv` 입니다. |
| 8 | `from openai import OpenAI` — OpenAI API를 부르는 클라이언트 클래스. |
| 10 | `load_dotenv()` — 작업 폴더(sec02)부터 위쪽 폴더로 올라가며 `.env` 를 찾아 `OPENAI_API_KEY` 를 환경 변수로 올립니다. 못 찾아도 오류는 나지 않고 `False` 만 돌려줍니다. |
| 11 | `client = OpenAI()` — 클라이언트 생성. `api_key` 를 생략하면 환경 변수 `OPENAI_API_KEY` 를 스스로 읽습니다. 키가 없으면 **이 줄에서** `OpenAIError: Missing credentials` 가 납니다 → 실행 전 확인 2번을 보세요. SDK는 429·연결 오류 때 스스로 2번까지 재시도합니다(기본값). |
| 12 | `MODEL = "gpt-4o"` — 쓸 모델 이름을 상수로. 다른 모델로 바꿀 때 이 한 곳만 고치면 됩니다. (2026-09 기준 `gpt-4o` 사용 가능) |
| 13 | `IMAGE_DIR = "../data/images"` — 사진 폴더. `..` 은 한 단계 위(sec02 → chap06)라서 실제 위치는 `chap06/data/images`. 결과 파일(md·json)도 이 폴더에 저장됩니다. |

In [1]:
import base64
import json
import mimetypes
import os
import re
from glob import glob
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-4o"
IMAGE_DIR = "../data/images"          # 노트북이 sec02 폴더에 있다는 전제 (VS Code 기본 동작)

## 셀 2 — 문제 출제 지시문(프롬프트)과 영어 추출 패턴

모델에게 보낼 지시문입니다. 말로 형식을 길게 설명하는 대신 **완성된 예시 문제 하나**를 보여 주는 방식(one-shot)입니다. 예시의 내용(베이커리·기차)은 실제 사진과 상관없고 **형식만** 가르칩니다. 예시에 나오는 표지 글자 `Listening:` 과 `정답:` 이 셀 4에서 영어 문제를 잘라 내는 기준이 됩니다.

<details><summary>줄번호 붙은 코드 보기 (설명 표와 맞춰 볼 때)</summary>

```text
  1  QUIZ_PROMPT = """
  2  제공된 이미지를 바탕으로, 다음과 같은 양식으로 퀴즈를 만들어주세요.
  3  정답은 1~4 중 하나만 해당하도록 출제하세요.
  4  토익 리스닝 문제 스타일로 문제를 만들어주세요.
  5  아래는 예시입니다.
  6  ----- 예시 -----
  7  
  8  Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
  9  - (1) 베이커리에서 사람들이 빵을 사고 있는 모습이 담겨 있습니다.
 10  - (2) 맨 앞에 서 있는 사람은 빨간색 셔츠를 입고 있습니다.
 11  - (3) 기차를 타기 위해 줄을 서 있는 사람들이 있습니다.
 12  - (4) 점원은 노란색 티셔츠를 입고 있습니다.
 13  
 14  Listening: Which of the following descriptions of the image is incorrect?
 15  - (1) It shows people buying bread at a bakery.
 16  - (2) The person standing at the front is wearing a red shirt.
 17  - (3) There are people lining up to take a train.
 18  - (4) The clerk is wearing a yellow T-shirt.
 19  
 20  정답: (4) 점원은 노란색 티셔츠가 아닌 파란색 티셔츠를 입고 있습니다.
 21  (주의: 정답은 1~4 중 하나만 선택되도록 출제하세요.)
 22  ======
 23  """
 24  
 25  ENG_PATTERN = re.compile(r"Listening:\s*(.*?)\s*정답:", re.S)   # 공백이 없어도, 줄이 바뀌어도 잡음
```
</details>

| 줄 | 설명 |
|---:|---|
| 1 | `QUIZ_PROMPT = """` — 따옴표 3개로 시작하는 **여러 줄 문자열**. 1~23줄 전체가 문자열 하나입니다. 원본은 함수 안에 있어 들여쓰기 공백 4칸이 프롬프트에 섞였지만, 여기서는 함수 밖 상수라 공백이 없습니다. 이름을 대문자로 쓴 것은 “바꾸지 않는 값(상수)”이라는 파이썬 관례입니다. |
| 2 | 지시 ① 무엇을 할지: 사진을 보고 아래 양식으로 퀴즈를 만들라. |
| 3 | 지시 ② 정답 조건: 정답이 하나만 되도록. 정답이 둘이거나 없는 문제를 막습니다. |
| 4 | 지시 ③ 문제 스타일: 토익 리스닝처럼 짧고 들어서 이해할 수 있는 문장으로. |
| 5 | 다음 부분이 **형식 견본**이라고 알려 줍니다. |
| 6 | `----- 예시 -----` — 지시와 예시를 나누는 구분선. 모델이 경계를 알아보기 쉽게 합니다. |
| 8 | 예시 한국어 질문 줄. `Q:` 로 시작하는 형식을 보여 줍니다. |
| 9 | 예시 보기 (1) — 사진과 맞는 설명의 예. `- (번호)` 형식을 보여 줍니다. |
| 10 | 예시 보기 (2) — 맞는 설명의 예. |
| 11 | 예시 보기 (3) — 맞는 설명의 예. |
| 12 | 예시 보기 (4) — **틀린 설명**의 예(정답이 될 보기). |
| 14 | 영어 질문 줄. **`Listening:`** 이 영어 문제의 시작 표지입니다. 셀 2 마지막 줄의 정규식이 이 글자를 찾습니다. |
| 15 | 영어 보기 (1) — 한국어 보기 (1)의 번역. 한국어·영어 문제가 **같은 보기, 같은 정답**이 되도록 번역 관계로 보여 줍니다. |
| 16 | 영어 보기 (2). |
| 17 | 영어 보기 (3). |
| 18 | 영어 보기 (4). |
| 20 | 예시 정답 줄. **`정답:`** 이 영어 문제의 끝 표지입니다. 번호뿐 아니라 틀린 이유도 쓰게 해서 해설로 쓸 수 있습니다. |
| 21 | 정답 조건(지시 ②)을 한 번 더 강조. 중요한 조건은 끝에 반복하면 더 잘 지켜집니다. |
| 22 | `======` — 예시의 끝 표시. |
| 23 | `"""` — 여러 줄 문자열 끝. |
| 25 | `ENG_PATTERN = re.compile(r"Listening:\s*(.*?)\s*정답:", re.S)` — 영어 문제를 찾을 패턴을 미리 만들어(컴파일) 둡니다. `r"..."` 는 `\` 를 글자 그대로 쓰는 문자열. `Listening:` 다음 `\s*` 는 공백·줄바꿈이 **0개 이상**(원본은 공백 1칸이 꼭 있어야 했음), `(.*?)` 는 가장 짧게 맞는 부분을 **그룹 1**로 잡고, 다시 `\s*` 뒤에 `정답:` 이 와야 합니다. `re.S` 는 `.` 이 줄바꿈도 포함하게 해서 여러 줄짜리 영어 문제를 한 번에 잡습니다. |

In [2]:
QUIZ_PROMPT = """
제공된 이미지를 바탕으로, 다음과 같은 양식으로 퀴즈를 만들어주세요.
정답은 1~4 중 하나만 해당하도록 출제하세요.
토익 리스닝 문제 스타일로 문제를 만들어주세요.
아래는 예시입니다.
----- 예시 -----

Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 베이커리에서 사람들이 빵을 사고 있는 모습이 담겨 있습니다.
- (2) 맨 앞에 서 있는 사람은 빨간색 셔츠를 입고 있습니다.
- (3) 기차를 타기 위해 줄을 서 있는 사람들이 있습니다.
- (4) 점원은 노란색 티셔츠를 입고 있습니다.

Listening: Which of the following descriptions of the image is incorrect?
- (1) It shows people buying bread at a bakery.
- (2) The person standing at the front is wearing a red shirt.
- (3) There are people lining up to take a train.
- (4) The clerk is wearing a yellow T-shirt.

정답: (4) 점원은 노란색 티셔츠가 아닌 파란색 티셔츠를 입고 있습니다.
(주의: 정답은 1~4 중 하나만 선택되도록 출제하세요.)
======
"""

ENG_PATTERN = re.compile(r"Listening:\s*(.*?)\s*정답:", re.S)   # 공백이 없어도, 줄이 바뀌어도 잡음

## 셀 3 — 사진 파일을 데이터 URL로 바꾸는 함수

사진을 `data:<형식>;base64,<본문>` 모양의 문자열로 바꿉니다. 이 문자열을 요청의 `image_url` 에 넣으면 인터넷에 올리지 않은 내 PC의 사진도 보낼 수 있습니다.

<details><summary>줄번호 붙은 코드 보기 (설명 표와 맞춰 볼 때)</summary>

```text
  1  def to_data_url(image_path):
  2      mime, _ = mimetypes.guess_type(image_path)
  3      if mime is None:
  4          raise ValueError(f"이미지 형식을 알 수 없습니다: {image_path}")
  5      with open(image_path, "rb") as f:
  6          return f"data:{mime};base64," + base64.b64encode(f.read()).decode("utf-8")
```
</details>

| 줄 | 설명 |
|---:|---|
| 1 | `def to_data_url(image_path):` — 함수 정의. 사진 경로(문자열)를 받아 데이터 URL(문자열)을 돌려줍니다. 이 줄을 실행하면 함수가 **등록만** 되고 아직 아무 파일도 읽지 않습니다. |
| 2 | `mime, _ = mimetypes.guess_type(image_path)` — 확장자로 형식을 알아냅니다. 돌려주는 값이 `('image/jpeg', None)` 처럼 **두 개**(형식, 압축 방식)라서 두 변수로 나눠 받고, 필요 없는 두 번째는 `_` 에 버립니다. 원본은 PNG도 `image/jpeg` 로 적었는데, 이 줄로 해결됩니다. |
| 3 | `if mime is None:` — 모르는 확장자(예: `.heic`)면 `None` 이 나옵니다. |
| 4 | `raise ValueError(...)` — 형식을 모르는 파일은 보내지 않고 바로 오류를 냅니다. 서버까지 갔다가 오류가 나는 것보다 원인을 알기 쉽습니다. (이 노트북은 `*.jpg` 만 다루므로 실제로 여기에 오지 않습니다.) |
| 5 | `with open(image_path, "rb") as f:` — 파일을 **바이트 읽기**(`r`=읽기, `b`=바이너리)로 엽니다. `with` 블록이 끝나면 파일이 자동으로 닫힙니다. |
| 6 | `return f"data:{mime};base64," + base64.b64encode(f.read()).decode("utf-8")` — ① `f.read()` 파일 전체를 bytes로 → ② `b64encode` 로 base64 bytes로(크기 약 4/3배) → ③ `.decode("utf-8")` 로 문자열로 → ④ 앞에 `data:image/jpeg;base64,` 를 붙여 돌려줍니다. |

In [3]:
def to_data_url(image_path):
    mime, _ = mimetypes.guess_type(image_path)
    if mime is None:
        raise ValueError(f"이미지 형식을 알 수 없습니다: {image_path}")
    with open(image_path, "rb") as f:
        return f"data:{mime};base64," + base64.b64encode(f.read()).decode("utf-8")

## 셀 4 — 답변에서 영어 문제만 꺼내는 함수

모델의 답변(한국어 문제 + 영어 문제 + 정답)에서 `Listening:` 과 `정답:` 사이만 잘라 냅니다. 형식이 틀리면 오류를 내지 않고 `None` 을 돌려줘서, 셀 5가 **다시 요청할지 판단**하는 데 씁니다.

<details><summary>줄번호 붙은 코드 보기 (설명 표와 맞춰 볼 때)</summary>

```text
  1  def extract_english(content):
  2      """'Listening:' 과 '정답:' 사이의 영어 문제만 꺼냄. 형식이 틀리면 None"""
  3      m = ENG_PATTERN.search(content)
  4      return m.group(1).strip() if m else None
```
</details>

| 줄 | 설명 |
|---:|---|
| 1 | `def extract_english(content):` — 답변 문자열을 받는 함수 정의. |
| 2 | 함수 설명문(docstring). 함수 바로 아래 문자열은 `help(extract_english)` 나 VS Code 마우스 올림 설명에 나타납니다. |
| 3 | `m = ENG_PATTERN.search(content)` — 셀 2의 패턴으로 답변 전체에서 **처음 맞는 곳**을 찾습니다. 찾으면 Match 객체, 못 찾으면 `None`. |
| 4 | `return m.group(1).strip() if m else None` — 한 줄 if(조건 표현식). 찾았으면 `group(1)`(패턴의 괄호 부분 = 영어 질문과 보기 4줄)에서 앞뒤 공백을 지워 돌려주고, 못 찾았으면 `None`. 원본의 `split(...)[1]` 은 표지가 없으면 `IndexError` 로 멈췄지만, 이 방식은 멈추지 않습니다. 표지 글자 `Listening:` 자체는 결과에 들어가지 않으므로 `Which of ...` 로 시작합니다. |

In [4]:
def extract_english(content):
    """'Listening:' 과 '정답:' 사이의 영어 문제만 꺼냄. 형식이 틀리면 None"""
    m = ENG_PATTERN.search(content)
    return m.group(1).strip() if m else None

## 셀 5 — 사진 한 장으로 문제 만들기 (최대 3번 시도)

핵심 함수입니다. 원본은 실패하면 자기 자신을 다시 부르는 **재귀**였고, 오류 출력 줄에서 `TypeError` 가 나거나 3번째 실패에서 예외를 던져 셀 전체가 멈췄습니다. 여기서는 `for` 반복문으로 최대 3번 시도하고, 끝내 실패하면 **`False` 를 돌려줄 뿐 멈추지 않습니다**.

<details><summary>줄번호 붙은 코드 보기 (설명 표와 맞춰 볼 때)</summary>

```text
  1  def image_quiz(image_path, max_trial=3):
  2      """성공하면 (문제 전문, 영어 문제, True), 끝내 실패하면 (None, None, False)"""
  3      messages = [{
  4          "role": "user",
  5          "content": [
  6              {"type": "text", "text": QUIZ_PROMPT},
  7              {"type": "image_url", "image_url": {"url": to_data_url(image_path)}},
  8          ],
  9      }]
 10      for n_trial in range(1, max_trial + 1):            # 재귀 대신 반복문: 횟수가 눈에 보임
 11          try:
 12              response = client.chat.completions.create(model=MODEL, messages=messages)
 13          except Exception as e:
 14              print(f"  [{n_trial}/{max_trial}] 호출 실패: {type(e).__name__}: {e}")   # 문자열로 변환해서 출력
 15              continue
 16          content = response.choices[0].message.content or ""
 17          eng = extract_english(content)
 18          if eng:
 19              return content, eng, True
 20          print(f"  [{n_trial}/{max_trial}] 형식이 맞지 않아 다시 요청합니다.")
 21      return None, None, False
```
</details>

| 줄 | 설명 |
|---:|---|
| 1 | `def image_quiz(image_path, max_trial=3):` — 사진 경로를 받는 함수. `max_trial=3` 은 **기본값 인자**로, 호출할 때 생략하면 3번 시도합니다(`image_quiz(g, max_trial=5)` 처럼 바꿀 수 있음). 원본에 있던 `n_trial` 인자는 반복문 안으로 들어가 필요 없어졌습니다. |
| 2 | 함수 설명문: 돌려주는 값이 항상 **3개짜리 튜플**이라는 약속. 성공·실패 모두 모양이 같아 호출하는 쪽(셀 6)이 같은 방식으로 받을 수 있습니다. |
| 3 | `messages = [{` — 요청 메시지(대화 기록) 시작. 리스트 안에 말 하나(딕셔너리)를 넣습니다. **반복문 밖에서 한 번만** 만들므로 재시도할 때 사진을 다시 읽고 인코딩하지 않습니다(원본은 매번 다시 했음). |
| 4 | `"role": "user",` — 사용자가 한 말이라는 표시. |
| 5 | `"content": [` — 글과 이미지를 섞어 보내므로 내용을 **조각의 리스트**로 씁니다. |
| 6 | `{"type": "text", "text": QUIZ_PROMPT},` — 첫 조각: 셀 2의 지시문. |
| 7 | `{"type": "image_url", "image_url": {"url": to_data_url(image_path)}},` — 둘째 조각: 셀 3 함수로 만든 데이터 URL(사진 전체). `detail` 을 생략해 `auto` 로 처리됩니다. |
| 8 | `],` — 조각 리스트 닫기. |
| 9 | `}]` — 말 딕셔너리와 messages 리스트를 함께 닫기. |
| 10 | `for n_trial in range(1, max_trial + 1):` — `range(1, 4)` 는 1, 2, 3을 차례로 만듭니다(끝 값 4는 포함 안 됨). 그래서 `n_trial` 이 그대로 “몇 번째 시도”가 됩니다. |
| 11 | `try:` — 다음 줄의 API 호출에서 오류가 나도 함수가 멈추지 않게 감쌉니다. |
| 12 | `response = client.chat.completions.create(model=MODEL, messages=messages)` — GPT-4o에 요청하고 답을 기다립니다. 이 안에서 SDK가 429·연결 오류 때 먼저 2번까지 스스로 재시도하고, 그래도 실패하면 예외를 13줄로 넘깁니다. |
| 13 | `except Exception as e:` — 12줄에서 난 오류(요청 한도 429, 인증 401, 연결 끊김 등)를 받아 `e` 에 담습니다. |
| 14 | `print(f"  [{n_trial}/{max_trial}] 호출 실패: {type(e).__name__}: {e}")` — 예: `[1/3] 호출 실패: RateLimitError: ...`. **f-문자열 안의 `{e}` 는 자동으로 문자열로 바뀝니다**. 원본의 `"failed\n" + e` 는 문자열과 예외 객체를 더하려다 `TypeError` 가 났던 부분입니다. `type(e).__name__` 은 오류 종류 이름입니다. |
| 15 | `continue` — 이번 시도는 포기하고 반복문의 다음 차례(다음 시도)로 넘어갑니다. |
| 16 | `content = response.choices[0].message.content or ""` — 답변 본문. 드물게 `None` 이 올 수 있는데, `or ""` 를 붙이면 `None` 일 때 빈 문자열로 바뀌어 다음 줄에서 오류가 나지 않습니다. |
| 17 | `eng = extract_english(content)` — 셀 4 함수로 영어 문제를 꺼냅니다. 형식이 맞으면 문자열, 틀리면 `None`. |
| 18 | `if eng:` — 영어 문제를 꺼냈다면 성공. 원본은 `"Listening:" in content` 로 검사하고 다른 방식으로 잘라서 “검사는 통과, 자르기는 실패”가 생길 수 있었지만, 여기서는 **검사와 추출이 같은 동작**이라 그런 일이 없습니다. |
| 19 | `return content, eng, True` — 성공: (문제 전문, 영어 문제, True)를 돌려주고 함수를 바로 끝냅니다. 남은 시도는 하지 않습니다. |
| 20 | `print(...형식이 맞지 않아 다시 요청합니다.)` — 답은 왔지만 양식이 틀린 경우. 출력 후 반복문의 다음 차례로 넘어가 다시 요청합니다. |
| 21 | `return None, None, False` — 반복문이 3번 다 돌도록 성공하지 못했을 때만 이 줄에 옵니다. 예외를 던지지 않고 실패 표시만 돌려주므로 **셀 6의 반복문이 멈추지 않습니다**. |

In [5]:
def image_quiz(image_path, max_trial=3):
    """성공하면 (문제 전문, 영어 문제, True), 끝내 실패하면 (None, None, False)"""
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": QUIZ_PROMPT},
            {"type": "image_url", "image_url": {"url": to_data_url(image_path)}},
        ],
    }]
    for n_trial in range(1, max_trial + 1):            # 재귀 대신 반복문: 횟수가 눈에 보임
        try:
            response = client.chat.completions.create(model=MODEL, messages=messages)
        except Exception as e:
            print(f"  [{n_trial}/{max_trial}] 호출 실패: {type(e).__name__}: {e}")   # 문자열로 변환해서 출력
            continue
        content = response.choices[0].message.content or ""
        eng = extract_english(content)
        if eng:
            return content, eng, True
        print(f"  [{n_trial}/{max_trial}] 형식이 맞지 않아 다시 요청합니다.")
    return None, None, False

## 셀 6 — 모든 사진으로 문제집 만들기 (여기서 API를 부릅니다)

jpg 사진마다 셀 5를 불러 문제를 만들고, 성공한 것만 번호를 매겨 md와 json으로 저장합니다. 원본 `.py` 의 `if __name__ == "__main__":` 블록입니다. 노트북에서는 그 조건이 항상 참이라 빼고 들여쓰기를 한 단계 줄였습니다(동작은 같음).

<details><summary>줄번호 붙은 코드 보기 (설명 표와 맞춰 볼 때)</summary>

```text
  1  txt = ""
  2  eng_list = []
  3  no = 1
  4  for g in sorted(glob(f"{IMAGE_DIR}/*.jpg")):          # 정렬해서 어느 PC에서나 같은 번호
  5      filename = os.path.basename(g)
  6      print(f"{filename} 출제 중...")
  7      q, eng, ok = image_quiz(g)
  8      if not ok:
  9          print(f"  → {filename} 은(는) 건너뜁니다.")
 10          continue
 11  
 12      txt += f"## 문제 {no}\n\n![image]({filename})\n\n{q}\n\n---------------------\n\n"
 13      eng_list.append({"no": no, "eng": eng, "img": filename})
 14  
 15      # 매 문제마다 저장: 중간에 멈춰도 그때까지의 결과가 남음
 16      with open(f"{IMAGE_DIR}/image_quiz_eng.md", "w", encoding="utf-8") as f:
 17          f.write(txt)
 18      with open(f"{IMAGE_DIR}/image_quiz_eng.json", "w", encoding="utf-8") as f:
 19          json.dump(eng_list, f, ensure_ascii=False, indent=4)
 20      no += 1
 21  
 22  print(f"완료: {no - 1}문제 저장")
```
</details>

| 줄 | 설명 |
|---:|---|
| 1 | `txt = ""` — 마크다운 문제집 전체를 이어 붙여 나갈 빈 문자열. |
| 2 | `eng_list = []` — 영어 문제를 `{번호, 영어 문제, 사진 이름}` 딕셔너리로 모을 빈 리스트. 원본 이름 `eng_dict` 는 실제로 리스트라 헷갈려서 바꿨습니다(내용·저장 형식은 같음). |
| 3 | `no = 1` — 문제 번호. **성공한 문제에만** 1씩 올리므로 건너뛴 사진이 있어도 번호에 빈칸이 생기지 않습니다. |
| 4 | `for g in sorted(glob(f"{IMAGE_DIR}/*.jpg")):` — `../data/images` 의 jpg 경로를 **이름순으로 정렬**해 하나씩 `g` 에 담아 반복합니다(8장). 정렬하지 않으면 운영체제에 따라 순서가 달라 문제 번호와 mp3 번호가 PC마다 바뀔 수 있습니다. png 그래프는 `*.jpg` 에 안 맞아 제외됩니다. |
| 5 | `filename = os.path.basename(g)` — 경로에서 파일 이름만: `'../data/images/busan_dive.jpg'` → `'busan_dive.jpg'`. 윈도우의 `\` 가 섞인 경로도 처리합니다. |
| 6 | `print(f"{filename} 출제 중...")` — 진행 상황 표시. 사진 한 장에 몇 초씩 걸리므로 어디까지 왔는지 보여 줍니다. |
| 7 | `q, eng, ok = image_quiz(g)` — 셀 5 함수 호출(**API 호출 1~3번**). 돌려받은 튜플 3개 값을 세 변수에 나눠 담습니다(언패킹). |
| 8 | `if not ok:` — 3번 모두 실패한 경우. |
| 9 | `print(f"  → {filename} 은(는) 건너뜁니다.")` — 어떤 사진을 건너뛰었는지 알려 줍니다. |
| 10 | `continue` — 아래 저장 단계를 건너뛰고 다음 사진으로. `no` 도 올라가지 않습니다. 원본은 이 경우 셀 전체가 멈췄습니다. |
| 12 | `txt += f"## 문제 {no}\n\n![image]({filename})\n\n{q}\n\n---------------------\n\n"` — 마크다운 한 문제 분량을 덧붙입니다: ① `## 문제 1` 제목 ② `![image](busan_dive.jpg)` 사진 링크(md 파일이 사진과 같은 폴더라 이름만 써도 보임) ③ 문제 전문 `q` ④ 구분선. `\n\n` 은 빈 줄입니다. 원본의 세 번에 나눈 `+=` 를 한 줄로 합쳤습니다. |
| 13 | `eng_list.append({"no": no, "eng": eng, "img": filename})` — 영어 문제 하나를 리스트에 추가. `no` 는 tts에서 mp3 파일 이름과 첫마디(Number 1)에, `eng` 는 읽을 내용으로, `img` 는 어떤 사진의 문제인지 연결하는 데 씁니다. 원본과 같은 키 이름이라 `tts.ipynb` 를 고치지 않아도 됩니다. |
| 15 | 주석: 왜 반복문 **안에서** 저장하는지 — 중간에 오류나 중단이 있어도 그때까지 만든 문제가 파일에 남습니다. |
| 16 | `with open(f"{IMAGE_DIR}/image_quiz_eng.md", "w", encoding="utf-8") as f:` — md 파일을 쓰기 모드로 엽니다. `"w"` 는 **내용을 지우고 새로** 쓰지만, `txt` 에 지금까지의 문제가 모두 있으므로 결과적으로 한 문제씩 늘어납니다. `encoding="utf-8"` 이 없으면 윈도우 기본 인코딩(cp949)으로 저장되어 `café` 같은 글자에서 오류가 날 수 있습니다. |
| 17 | `f.write(txt)` — 누적 문자열 전체를 씁니다. |
| 18 | `with open(f"{IMAGE_DIR}/image_quiz_eng.json", "w", encoding="utf-8") as f:` — json 파일을 쓰기 모드로 엽니다. |
| 19 | `json.dump(eng_list, f, ensure_ascii=False, indent=4)` — 리스트를 JSON으로 저장. `ensure_ascii=False` 는 한글·é를 그대로(`\ud55c` 같은 코드로 바꾸지 않고), `indent=4` 는 4칸 들여쓰기로 보기 좋게 저장합니다. |
| 20 | `no += 1` — 다음 문제 번호. 성공해서 저장까지 끝난 경우에만 실행됩니다. |
| 22 | `print(f"완료: {no - 1}문제 저장")` — 반복이 끝난 뒤 저장한 문제 수를 알려 줍니다. `no` 는 “다음 번호”라서 1을 뺍니다. 8장 모두 성공하면 `완료: 8문제 저장`. |

In [6]:
txt = ""
eng_list = []
no = 1
for g in sorted(glob(f"{IMAGE_DIR}/*.jpg")):          # 정렬해서 어느 PC에서나 같은 번호
    filename = os.path.basename(g)
    print(f"{filename} 출제 중...")
    q, eng, ok = image_quiz(g)
    if not ok:
        print(f"  → {filename} 은(는) 건너뜁니다.")
        continue

    txt += f"## 문제 {no}\n\n![image]({filename})\n\n{q}\n\n---------------------\n\n"
    eng_list.append({"no": no, "eng": eng, "img": filename})

    # 매 문제마다 저장: 중간에 멈춰도 그때까지의 결과가 남음
    with open(f"{IMAGE_DIR}/image_quiz_eng.md", "w", encoding="utf-8") as f:
        f.write(txt)
    with open(f"{IMAGE_DIR}/image_quiz_eng.json", "w", encoding="utf-8") as f:
        json.dump(eng_list, f, ensure_ascii=False, indent=4)
    no += 1

print(f"완료: {no - 1}문제 저장")

busan_dive.jpg 출제 중...
local_stitch.jpg 출제 중...
local_stitch_terrarosa.jpg 출제 중...
mangwon.jpg 출제 중...
mangwon_bakery.jpg 출제 중...
sangam_interior.jpg 출제 중...
  [1/3] 형식이 맞지 않아 다시 요청합니다.
seolleung_terrarosa.jpg 출제 중...
stanford_coffee.jpg 출제 중...
완료: 8문제 저장


## 셀 7 — 결과 확인과 다음 단계

- `chap06/data/images/image_quiz_eng.md` 를 VS Code에서 열고 **미리보기**(`Ctrl+Shift+V`)를 누르면 사진과 문제가 함께 보입니다.
- `chap06/data/images/image_quiz_eng.json` 에는 `{"no", "eng", "img"}` 항목이 문제 수만큼 들어 있습니다. `eng` 는 `Which of the following ...` 로 시작합니다.
- 다음은 `tts.ipynb` 에서 이 json을 읽어 mp3를 만듭니다. 그 전에 `chap06/data/audio` 폴더가 있는지 확인하세요(없으면 `os.makedirs("../data/audio", exist_ok=True)`).
- 모델이 사진을 잘못 볼 수도 있으니, 시험에 쓰기 전에 **사람이 한 번 검수**하세요.

| 출력 예 | 뜻 |
|---|---|
| `busan_dive.jpg 출제 중...` | 이 사진의 문제를 요청하는 중 |
| `  [1/3] 호출 실패: RateLimitError: ...` | 1번째 시도에서 요청 한도 초과 → 자동으로 2번째 시도 |
| `  [2/3] 형식이 맞지 않아 다시 요청합니다.` | 답은 왔지만 `Listening:`/`정답:` 형식이 아님 → 다시 요청 |
| `  → local_stitch.jpg 은(는) 건너뜁니다.` | 3번 모두 실패 → 이 사진만 빼고 계속 |
| `완료: 8문제 저장` | 끝. 저장된 문제 수 |